# Portfolio Optimization (Monte Carlo + MVO)
This notebook contains all functions (Monte‑Carlo simulation, multi‑start Mean‑Variance optimisation, etc.) in separate cells so they compile **once**. Then you can benchmark the end‑to‑end run time with `%time` / `%timeit` without the function‑definition overhead.

In [ ]:
# === monte_carlo.py ===
import numpy as np
import pandas as pd
import time
from joblib import Parallel, delayed
import multiprocessing

def _simulate_single_run(mean_returns, cov_matrix, rf, n):
    w = np.random.random(n)
    w /= w.sum()
    ret = w @ mean_returns
    vol = (w.T @ cov_matrix @ w) ** 0.5
    sharpe = (ret - rf) / vol
    return w, ret, vol, sharpe

def monte_carlo_portfolio_optimization_opt(
    df: pd.DataFrame,
    n_simulations: int = 100_000,
    n_assets_to_select: int = 50,
    risk_free_rate: float = 0.02,
    random_seed: int = 42,
    fixed_tickers=None
):
    """Fast, multi‑processed Monte‑Carlo search for the best Sharpe‑ratio portfolio."""
    t0 = time.time()
    np.random.seed(random_seed)

    # --- choose universe ---------------------------------------------------------
    if fixed_tickers is not None:
        selected = fixed_tickers
    else:
        full = df.columns.to_list()
        if len(full) < n_assets_to_select:
            raise ValueError("Asset pool smaller than number to select")
        selected = np.random.choice(full, size=n_assets_to_select, replace=False)

    sub = df[selected]
    mean_ret = sub.mean().values * 252
    cov = sub.cov().values * 252
    n = len(selected)

    # --- simulate in parallel ----------------------------------------------------
    n_jobs = multiprocessing.cpu_count()
    res = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_simulate_single_run)(mean_ret, cov, risk_free_rate, n)
        for _ in range(n_simulations)
    )
    weights, rets, vols, sharpes = map(np.array, zip(*res))
    idx = sharpes.argmax()

    print(f"Monte‑Carlo (parallel) runtime: {time.time()-t0:.2f}s")
    return {
        "tickers": selected,
        "max_sharpe": sharpes[idx],
        "expected_return": rets[idx],
        "expected_volatility": vols[idx],
        "optimal_weights": weights[idx],
        "returns": rets,
        "volatilities": vols,
        "sharpes": sharpes
    }


In [ ]:
# === optimize_mvo.py ===
import numpy as np
import pandas as pd
from numba import njit
from scipy.optimize import minimize, Bounds, LinearConstraint

@njit
def _neg_sharpe(weights, mean_returns, cov_matrix, rf):
    ret = weights @ mean_returns
    vol = (weights.T @ cov_matrix @ weights) ** 0.5
    return -(ret - rf) / vol

def _weight_constraints(n):
    return Bounds(0, 1), LinearConstraint(np.ones(n), lb=1, ub=1)

def optimize_portfolio_from_weights(csv_path, init_weights, risk_free_rate=0.02, tickers=None):
    """SLSQP MVO starting from given weights, using *pre‑computed* mean/Σ stored on disk."""
    mean_df = pd.read_csv("./data/excess_mean.csv", index_col=0)
    cov_df  = pd.read_csv("./data/excess_cov.csv",  index_col=0)

    if tickers is not None:
        mu = mean_df.loc[tickers].squeeze().values
        cov = cov_df.loc[tickers, tickers].values
    else:
        mu = mean_df.squeeze().values
        cov = cov_df.values
        tickers = mean_df.index.to_list()

    bounds, constr = _weight_constraints(len(mu))

    res = minimize(_neg_sharpe, x0=np.array(init_weights),
                   args=(mu, cov, risk_free_rate),
                   bounds=bounds, constraints=[constr],
                   method='SLSQP', options={'disp': False})

    w_opt = res.x
    return {
        "optimized_weights": w_opt,
        "sharpe": -res.fun,
        "expected_return": w_opt @ mu,
        "expected_volatility": (w_opt.T @ cov @ w_opt) ** 0.5,
        "tickers": tickers
    }


In [ ]:
# === orchestration ===
def run_portfolio_optimization(processed_csv_path: str,
                               num_trials: int = 50,
                               n_assets: int = 50,
                               top_k: int = 10) -> dict:
    """Run MC + multi‑start MVO pipeline and return best result dict."""
    returns = pd.read_csv(processed_csv_path, index_col=0, parse_dates=True)
    if 'SHY' not in returns.columns:
        raise ValueError("Ticker 'SHY' not in data")
    rf_series = returns['SHY']
    df = returns.drop(columns=['SHY'])

    best_mc = []
    for seed in range(num_trials):
        res = monte_carlo_portfolio_optimization_opt(
            df, n_simulations=5_000, n_assets_to_select=n_assets,
            risk_free_rate=rf_series.mean(), random_seed=seed)
        best_mc.append(res)
        best_mc = sorted(best_mc, key=lambda x: x['max_sharpe'], reverse=True)[:top_k]

    # refine MC on each of top_k ticker sets
    refined = [monte_carlo_portfolio_optimization_opt(
                   df, n_simulations=100_000,
                   n_assets_to_select=n_assets,
                   risk_free_rate=rf_series.mean(),
                   fixed_tickers=r['tickers'])
               for r in best_mc]

    # MVO starting from each refined weight vector
    mvo_results = [optimize_portfolio_from_weights(
                       csv_path=processed_csv_path,
                       init_weights=r['optimal_weights'],
                       tickers=r['tickers'],
                       risk_free_rate=rf_series.mean())
                   for r in refined]

    best = max(mvo_results, key=lambda x: x['sharpe'])
    return best


## Benchmark
Run the full pipeline and time it:

In [ ]:
%time result = run_portfolio_optimization('./data/raw/processed.csv')
result